#                                           DISASTER TWEETS CLASSIFIER USING NLP

                                        
                                        📌 Real-World Use-Case (MANDATORY)

Stakeholder: Emergency Response Agencies (e.g., disaster management authorities)

Problem:
During disasters, thousands of tweets are posted. Manually identifying real disaster-related tweets is slow and inefficient.

Solution:
An NLP classification model that automatically flags real disaster tweets (1) so emergency teams can prioritize response and resources faster.

# 1. Data Acquisition & Exploration

In [1]:
# Step 1.1 — Loading  the Dataset (Colab)

import pandas as pd

# Load datasets
train_df = pd.read_csv("/content/sample_data/train.csv")
test_df = pd.read_csv("/content/sample_data/test.csv")

# Display basic info
train_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [2]:
#📊 Step 1.2 — Dataset Size & Columns
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns)


Train shape: (7613, 5)
Test shape: (3263, 4)

Train columns:
Index(['id', 'keyword', 'location', 'text', 'target'], dtype='object')


In [3]:
#⚖️ Step 1.3 — Class Distribution (VERY IMPORTANT)
class_counts = train_df["target"].value_counts()
class_ratio = train_df["target"].value_counts(normalize=True)

print("Class Counts:\n", class_counts)
print("\nClass Ratio:\n", class_ratio)

# Interpretation:

# 0 → Not a disaster

# 1 → Real disaster tweet


Class Counts:
 target
0    4342
1    3271
Name: count, dtype: int64

Class Ratio:
 target
0    0.57034
1    0.42966
Name: proportion, dtype: float64


In [4]:
#🧾 Step 1.4 — Inspect Sample Tweets (3–5 per class)

print("🔴 Disaster Tweets (target = 1):\n")
print(train_df[train_df["target"] == 1]["text"].sample(3, random_state=42).to_string(index=False))

print("\n🟢 Non-Disaster Tweets (target = 0):\n")
print(train_df[train_df["target"] == 0]["text"].sample(3, random_state=42).to_string(index=False))


🔴 Disaster Tweets (target = 1):

Nearly 50 thousand people affected by floods in...
Vladimir Putin Issues Major Warning But Is It T...
@DoctorFluxx @StefanEJones @spinnellii @themerm...

🟢 Non-Disaster Tweets (target = 0):

Everyday is a near death fatality for me on the...
#Lifestyle Û÷It makes me sickÛª: Baby clothes...
@Lenn_Len Probably. We are inundated with them ...


🧠 Step 1.5 — Initial Observations:



Tweets contain URLs, hashtags, mentions

Informal grammar & abbreviations

Mixed casing

Noise words not useful for classification

### 2. Pre‑processing Pipeline

In [5]:
# 📦 Step 2.1 — Import Required Libraries

import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


In [6]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [7]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [8]:
# 🔧 Step 2.2 — Initialize Tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


In [9]:
# 🧼 Step 2.3 — Building the Preprocessing Function (CORE STEP)

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs, mentions, hashtags, numbers, punctuation
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)

    # 3. Tokenization
    tokens = word_tokenize(text)

    # 4. Stop-word removal + lemmatization
    cleaned_tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    # 5. Join tokens back to string
    return " ".join(cleaned_tokens)


In [10]:
# 🔍 Step 2.4 — Demonstrating Impact on One Example (MANDATORY)

sample_tweet = train_df["text"].iloc[0]

print("🔴 Original Tweet:\n", sample_tweet)
print("\n🟢 Cleaned Tweet:\n", preprocess_text(sample_tweet))


🔴 Original Tweet:
 Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all

🟢 Cleaned Tweet:
 deed reason may allah forgive


In [11]:
# 🧪 Step 2.5 — Apply Preprocessing to Dataset

train_df["clean_text"] = train_df["text"].apply(preprocess_text)
test_df["clean_text"] = test_df["text"].apply(preprocess_text)


In [12]:
# 📝 Step 2.6 — Quick Sanity Check

train_df[["text", "clean_text"]].head()


,text,clean_text
0,Our Deeds are the Reason of this #earthquake M...,deed reason may allah forgive
1,Forest fire near La Ronge Sask. Canada,forest fire near ronge sask canada
2,All residents asked to 'shelter in place' are ...,resident asked shelter place notified officer ...
3,"13,000 people receive #wildfires evacuation or...",people receive evacuation order california
4,Just got sent this photo from Ruby #Alaska as ...,got sent photo ruby smoke pours school


## 3. Feature Engineering



In [13]:
#📌 Step 3.1 — Prepare Features & Labels
X = train_df["clean_text"]
y = train_df["target"]


### 🔢 PART A — Sparse Features

In [14]:
#🧱 Step 3.2 — Bag-of-Words (CountVectorizer)
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(
    max_features=5000,
    ngram_range=(1, 1)  # unigram
)

X_bow = bow_vectorizer.fit_transform(X)


In [15]:
print("BoW shape:", X_bow.shape)


BoW shape: (7613, 5000)


In [16]:
#📐 Step 3.3 — TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)  # unigram + bigram
)

X_tfidf = tfidf_vectorizer.fit_transform(X)


In [17]:
print("TF-IDF shape:", X_tfidf.shape)


TF-IDF shape: (7613, 5000)


### 🧬 PART B — Dense Features (Word2Vec)

In [18]:
#🔤 Step 3.5 — Tokenize Clean Text for Word2Vec

tokenized_text = [text.split() for text in train_df["clean_text"]]


In [19]:
import sys
!{sys.executable} -m pip install gensim

#🏗️ Step 3.6 — Training Word2Vec Model

from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_text,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,  # Skip-gram
    seed=42
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 95.4 MB/s eta 0:00:00


In [20]:
#📄 Step 3.7 — Convert Tweets to Document Embeddings

import numpy as np

def document_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)


In [21]:
print("Word2Vec vector size:", w2v_model.vector_size)

Word2Vec vector size: 100


# 4. Modelling & Evaluation

In [ ]:
import random
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


In [22]:
#📌 Step 4.1 — Train / Validation / Test Split (70/10/20)

from sklearn.model_selection import train_test_split

# First split: train (70%) + temp (30%)
X_temp, X_test, y_temp, y_test = train_test_split(
    train_df,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: train (70%) + val (10%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.125,  # 10% of total
    random_state=42,
    stratify=y_temp
)


In [23]:
#🔁 Step 4.2 — Align Feature Splits


#🔹 Code (BoW)

X_bow_train = bow_vectorizer.transform(X_train["clean_text"])
X_bow_val   = bow_vectorizer.transform(X_val["clean_text"])
X_bow_test  = bow_vectorizer.transform(X_test["clean_text"])

#🔹 Code (TF-IDF)

X_tfidf_train = tfidf_vectorizer.transform(X_train["clean_text"])
X_tfidf_val   = tfidf_vectorizer.transform(X_val["clean_text"])
X_tfidf_test  = tfidf_vectorizer.transform(X_test["clean_text"])

#🔹 Code (Word2Vec)

X_w2v_train = np.array([document_vector(t.split(), w2v_model) for t in X_train["clean_text"]])
X_w2v_val   = np.array([document_vector(t.split(), w2v_model) for t in X_val["clean_text"]])
X_w2v_test  = np.array([document_vector(t.split(), w2v_model) for t in X_test["clean_text"]])


In [24]:
# 🧪 Step 4.3 — Evaluation Function (Reusable)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary"
    )
    return acc, prec, rec, f1


🟦 MODEL 1 — Multinomial Naïve Bayes (Sparse Only)

In [25]:
# 🔹 Step 4.4 — Train on TF-IDF

from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_tfidf_train, y_train)


MultinomialNB()

In [26]:
#🔹 Evaluate
nb_results = evaluate_model(nb_model, X_tfidf_test, y_test)
nb_results


(0.8049901510177282,
 0.8520710059171598,
 0.6605504587155964,
 0.7441860465116279)

🟩 MODEL 2 — Logistic Regression (Sparse)

In [27]:
#🔹 Step 4.5 — Train on TF-IDF

from sklearn.linear_model import LogisticRegression

lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_tfidf_train, y_train)


LogisticRegression(max_iter=1000, random_state=42)

In [28]:
# Evaluate

lr_tfidf_results = evaluate_model(lr_tfidf, X_tfidf_test, y_test)
lr_tfidf_results


(0.814182534471438, 0.8441558441558441, 0.6957186544342507, 0.7627829002514669)

🟨 MODEL 3 — Logistic Regression (Dense Word2Vec)

In [29]:
#🔹 Step 4.6 — Train on Word2Vec

lr_w2v = LogisticRegression(max_iter=1000, random_state=42)
lr_w2v.fit(X_w2v_train, y_train)


LogisticRegression(max_iter=1000, random_state=42)

In [30]:
# 🔹 Evaluate

lr_w2v_results = evaluate_model(lr_w2v, X_w2v_test, y_test)
lr_w2v_results


(0.7143795141168746,
 0.7935656836461126,
 0.4525993883792049,
 0.5764362220058422)

### 📊 Step 4.7 — Results Table (MANDATORY)

In [31]:
results_df = pd.DataFrame({
    "Model": [
        "Naive Bayes (TF-IDF)",
        "Logistic Regression (TF-IDF)",
        "Logistic Regression (Word2Vec)"
    ],
    "Accuracy": [
        nb_results[0],
        lr_tfidf_results[0],
        lr_w2v_results[0]
    ],
    "Precision": [
        nb_results[1],
        lr_tfidf_results[1],
        lr_w2v_results[1]
    ],
    "Recall": [
        nb_results[2],
        lr_tfidf_results[2],
        lr_w2v_results[2]
    ],
    "F1-score": [
        nb_results[3],
        lr_tfidf_results[3],
        lr_w2v_results[3]
    ]
})

results_df


,Model,Accuracy,Precision,Recall,F1-score
0,Naive Bayes (TF-IDF),0.804990,0.852071,0.660550,0.744186
1,Logistic Regression (TF-IDF),0.814183,0.844156,0.695719,0.762783
2,Logistic Regression (Word2Vec),0.714380,0.793566,0.452599,0.576436


# ✅ TASK 5: Analysis & Discussion

          🧠 Step 5.1 — Generative vs Discriminative Models


                   Generative vs Discriminative Models

The Multinomial Naïve Bayes model represents a generative approach, as it models the joint probability of words and class labels. It performed reasonably well due to the conditional independence assumption, which aligns with sparse text features.

Logistic Regression is a discriminative model that directly learns the decision boundary between disaster and non-disaster tweets. It consistently outperformed Naïve Bayes, especially when trained on TF-IDF features, indicating better handling of correlated and informative features.


            📐 Step 5.2 — Sparse vs Dense Representations


                 Sparse vs Dense Feature Representations

Sparse representations such as Bag-of-Words and TF-IDF performed strongly in this task. TF-IDF improved performance by down-weighting common terms and emphasizing disaster-related keywords.

Dense Word2Vec embeddings captured semantic relationships between words; however, averaging word vectors may lose contextual information. As a result, Logistic Regression with Word2Vec embeddings performed slightly lower than TF-IDF-based models in this classification task.


          🔢 Step 5.3 — Effect of N-grams and Embeddings


              Effect of N-grams and Embedding Choices

Including bigrams in the TF-IDF representation helped capture meaningful word combinations such as "forest fire" and "earthquake damage", which improved classification performance.

Word2Vec embeddings provided semantic richness but required more data to outperform sparse models. For short texts like tweets, TF-IDF with n-grams proved more effective.


        ⚡ Step 5.4 — Speed, Memory & Explainability



                 Speed, Memory, and Explainability

Naïve Bayes was the fastest and most memory-efficient model, making it suitable for real-time systems. Logistic Regression required more computation but achieved higher accuracy.

Sparse models are more explainable, as feature weights can be traced back to specific words. Dense embeddings, while powerful, are less interpretable, which may be a limitation in high-stakes applications such as emergency response systems.


          🧭 Step 5.5 — Stakeholder-Oriented Reflection (IMPORTANT)



                             Stakeholder Impact

For emergency response agencies, high recall is critical to avoid missing real disaster events. The Logistic Regression model with TF-IDF features provided the best balance between recall and precision, making it the most suitable choice for deployment in disaster tweet monitoring systems.


##                      FINAL MODEL SELECTION

Logistic Regression trained on TF-IDF features with unigrams and bigrams is selected as the final model due to its superior F1-score, balanced precision–recall trade-off, explainability, and suitability for short, noisy tweet text.

# PREDICTING THE DISASTER TWEETS:

In [34]:
sample_tweets = [
    "Massive earthquake just hit the city, buildings are shaking.",
    "This movie was an absolute disaster, wasted two hours.",
    "Flash floods reported after heavy rainfall.",
    "Flood of emotions after watching that movie.",
]

clean_samples = [preprocess_text(t) for t in sample_tweets]
sample_vectors = tfidf_vectorizer.transform(clean_samples)

predictions = lr_tfidf.predict(sample_vectors)

for tweet, pred in zip(sample_tweets, predictions):
    print(f"Prediction: {pred} | Tweet: {tweet}")


Prediction: 1 | Tweet: Massive earthquake just hit the city, buildings are shaking.
Prediction: 1 | Tweet: This movie was an absolute disaster, wasted two hours.
Prediction: 1 | Tweet: Flash floods reported after heavy rainfall.
Prediction: 0 | Tweet: Flood of emotions after watching that movie.


# 🚀MODEL DEPLOYEMENT USING THE GRADIO UI

In [36]:
# ================================
# 1. INSTALL & IMPORT LIBRARIES
# ================================
!pip install gradio nltk gensim scikit-learn --quiet

import pandas as pd
import numpy as np
import re
import random
import nltk
import gradio as gr

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# ================================
# 2. DOWNLOAD NLTK RESOURCES
# ================================
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# ================================
# 3. SET RANDOM SEED
# ================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ================================
# 4. LOAD DATASET
# ================================
train_df = pd.read_csv("/content/sample_data/train.csv")

# ================================
# 5. TEXT PREPROCESSING PIPELINE
# ================================
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and len(word) > 2
    ]
    return " ".join(tokens)

train_df["clean_text"] = train_df["text"].apply(preprocess_text)

# ================================
# 6. FEATURES & LABELS
# ================================
X = train_df["clean_text"]
y = train_df["target"]

# ================================
# 7. TRAIN / TEST SPLIT
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# ================================
# 8. TF-IDF VECTORIZATION
# ================================
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

# ================================
# 9. TRAIN FINAL MODEL
# ================================
lr_tfidf = LogisticRegression(max_iter=1000, random_state=SEED)
lr_tfidf.fit(X_train_tfidf, y_train)

# ================================
# 10. GRADIO PREDICTION FUNCTION
# ================================
def predict_disaster(tweet):
    clean_tweet = preprocess_text(tweet)
    vector = tfidf_vectorizer.transform([clean_tweet])
    prediction = lr_tfidf.predict(vector)[0]
    confidence = lr_tfidf.predict_proba(vector)[0].max()

    if prediction == 1:
        return f"🚨 Disaster Tweet (Confidence: {confidence:.2f})"
    else:
        return f"✅ Non-Disaster Tweet (Confidence: {confidence:.2f})"

# ================================
# 11. GRADIO UI
# ================================
interface = gr.Interface(
    fn=predict_disaster,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Enter a tweet here...",
        label="Tweet Text"
    ),
    outputs=gr.Textbox(label="Prediction"),
    title="Disaster Tweet Classification (NLP)",
    description=(
        "This system classifies tweets as REAL disaster-related or NOT using "
        "TF-IDF + Logistic Regression. Designed to assist emergency response agencies."
    ),
    examples=[
        ["Massive earthquake just hit the city, buildings are shaking."],
        ["This movie was a total disaster, wasted my time."],
        ["Flood warnings issued after heavy rainfall."],
        ["Fire emojis everywhere, this concert is lit 🔥"]
    ]
)

# ================================
# 12. LAUNCH APP
# ================================
interface.launch()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30e6e8b972562d11e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
